# COMPLETE PROJECT OBJECTIVE
(Extracted and consolidated from the project specification prompts)

## Mission
Build and demonstrate **MeshNet** as a test-driven, Colab-importable project (single `.ipynb` envelope, **Runtime → Run all**) that launches the whole development, testing, debug, simulation, and packaging flow. The project must preserve any existing work in the repository and produce/download launch-ready outputs: **APK build project**, **launchable HTML**, and **terminal-installable production package**.

## Network concept
- **Hamachi/Skype-like router-node-each infrastructure:** every participant is a node that can route/relay for others.
- **Clever use of all available communication channels:** Wi-Fi, Bluetooth, and near-neighbors sensing tricks (if allowed by the user), with channel abstraction/simulation and a real-transport integration path.
- **Density-aware capacity:** where higher usage and density exist, higher bandwidth/baudrate and stability are achieved (adaptive channel scoring/capacity model).
- **Resilience:** easily recognize and incorporate new nodes; maintain stability regardless of some nodes disappearing unexpectedly (reroute/heal).

## Connectivity and protocol
- **IPv6-first peer networking:** transfer contact signals using slow/bootstrap methods, then establish peer-to-peer **one-UDP-port**, **compression-encrypted** connections.
- **Slow/bootstrap contact channels:** QR code, NFC exchange, Bluetooth advertisement, manual paste, or local beacon/invite exchange carrying compact signed invitation material (node identity, nickname, endpoint/key info as appropriate).
- **No fixed ports:** do **not** hardcode ports (e.g. 8000). Bind UDP/HTTP to OS-assigned available ports (`port=0`) by default; invitations/UI carry the actual assigned endpoint. Optional **prime-port preference** may be applied within the dynamic/private range **49152–65535** (nearest/selected prime) as a preference only—**networking simplicity is paramount**.
- **No pi-sync / no time-server chain:** do not require Internet time servers or chained time sync. Use **monotonic clocks**, local timestamps, and peer **heartbeat/ACK** semantics for liveness/RTT/timeouts.
- **Transparent and simple:** work across networks where possible; **two connected nodes suffice** as a complete connection/mesh with **no Internet required**.
- **Routes, not location:** routes are visualized with common chat; physical location is secondary or of no importance.

## Identity, chat, and proof of communication
- **Peer-to-peer chat** showing ping distance (hops/path/latency) and **pregenerated/accepted unique nicknames**.
- **Common chat and invite/group chats** (emulated/supported across node types).
- **Hail on connect:** to prove communication, each newly connected node hails others in common chat with details (nickname, platform/type, node ID, transport/port/endpoint details, peers/context).
- **`?` in every client:** reveals the detailed complete objective and commands/help (terminal, browser/HTML, Android).
- **Blockchain of messages (common chat):** common chat is represented as an append-only, hash-linked message chain (index, timestamp, previous hash, hash, author, payload/kind/route metadata) so message history/integrity is visible in the running session.

## Platforms and outputs
- **Simulation:** simulate the whole network on several nodes of different type/platform (Android, bash/terminal, web/browser), demonstrating join, hail, chat, ping, groups, reroute, and stability.
- **Browser sessions:** several browser sessions can open the running session node/page, join, and participate; new nodes **smoothly pop in** on the **Sensei Network Map** (responsive animation).
- **Terminal:** terminal-installable production client/package (`meshctl`) for invite/join/chat/peers/routes/ping/serve-web/help.
- **Android:** one Android app/client path exchanging messages with browser/terminal nodes; provide buildable/launch-ready APK project (and compiled APK artifact when toolchain is available).
- **Website/HTML:** beacon/session HTML page displays common chat and blockchain, serves as joinable session node UI, and is provided as a launchable/downloadable artifact.

## Operational/runtime goals
- **Keep simulation running** and provide a testing link openable in several browsers to join and see the Blockchain of messages of common chat in the live HTML session node.
- **Sensei Network Map:** responsive animated network map where nodes/links update and newly joined nodes smoothly pop in.
- **Security:** signed invitations, authenticated/compression-aware encrypted envelopes, replay/sequence protection, local private keys, no sensitive data leakage in beacon/public views.
- **TDD discipline:** tests define/validate core behaviors (identity/invites/envelopes/compression/ports/routing/mesh/hail/blockchain/beacon/session APIs); implementation satisfies them.
- **Publishing:** publish all project materials to GitHub.


---
## How this notebook unfolds
1. **Foreground objective & `?` spec** (this section) — the implementation follows this objective.
2. **Install & scaffold** — dependencies and project structure.
3. **Core protocol & mesh** — objective/help, invites, envelopes, compression, port selection, routing, nodes, mesh, **hash-linked chat blockchain**, hails.
4. **Live session server & HTML** — multi-browser joinable beacon with **Blockchain of messages** view and **Sensei Network Map** responsive animation (smooth pop-in).
5. **Tests** — TDD validation of core and session behaviors.
6. **Keep-running simulation** — live session, node churn/join demo, and testing link (Colab proxy/local).
7. **Production artifacts** — launchable HTML, terminal-installable package, Android APK build project (and APK binary if toolchain available).
8. **GitHub publish** — commit/push project outputs.

> Existing repository work (e.g. `server/`) is preserved; this notebook adds `meshnet_project/`, `dist/`, and notebook artifacts.


In [ ]:
# 0) Environment & helpers
import os, sys, json, pathlib, subprocess, time, shutil, hashlib

IS_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG")) or "google.colab" in sys.modules
ROOT = pathlib.Path.cwd()
PROJECT = ROOT / "meshnet_project"
DIST = ROOT / "dist"
ARTIFACTS = pathlib.Path("/artifacts")
print("CWD:", ROOT)
print("IS_COLAB:", IS_COLAB)


In [ ]:
# 1) Install dependencies (Colab-safe)
def pip_install(pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + pkgs
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)

pkgs = ["pynacl>=1.5", "websockets>=12", "pytest>=8", "pytest-asyncio>=0.23"]
try:
    import nacl  # noqa
    import websockets  # noqa
    import pytest  # noqa
    import pytest_asyncio  # noqa
    print("Dependencies already available.")
except Exception:
    pip_install(pkgs)
    print("Dependencies installed.")


In [ ]:
# 2) Create scaffold
for p in [PROJECT / "meshnet", PROJECT / "tests", DIST, ROOT / "android_apk_project" / "app" / "src" / "main" / "java" / "com" / "meshnet"]:
    p.mkdir(parents=True, exist_ok=True)
if ARTIFACTS.parent.exists():
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("Scaffold ready:", PROJECT, DIST)


In [ ]:
%%writefile meshnet_project/meshnet/objective.py
"""Shared complete project objective and help text for all clients (terminal/web/Android)."""

FULL_OBJECTIVE = r"""COMPLETE PROJECT OBJECTIVE
(Extracted and consolidated from the project specification prompts)

MISSION
Build and demonstrate MeshNet as a test-driven, Colab-importable project (single .ipynb envelope, Runtime -> Run all) that launches the whole development, testing, debug, simulation, and packaging flow. Preserve existing repository work and produce/download launch-ready outputs: APK build project, launchable HTML, and terminal-installable production package.

NETWORK CONCEPT
- Hamachi/Skype-like router-node-each infrastructure: every participant is a node that can route/relay for others.
- Clever use of all available communication channels: Wi-Fi, Bluetooth, and near-neighbors sensing tricks (if allowed by the user), with channel abstraction/simulation and a real-transport integration path.
- Density-aware capacity: where higher usage and density exist, higher bandwidth/baudrate and stability are achieved (adaptive channel scoring/capacity model).
- Resilience: easily recognize and incorporate new nodes; maintain stability regardless of some nodes disappearing unexpectedly (reroute/heal).

CONNECTIVITY AND PROTOCOL
- IPv6-first peer networking: transfer contact signals using slow/bootstrap methods, then establish peer-to-peer one-UDP-port, compression-encrypted connections.
- Slow/bootstrap contact channels: QR code, NFC exchange, Bluetooth advertisement, manual paste, or local beacon/invite exchange carrying compact signed invitation material (node identity, nickname, endpoint/key info as appropriate).
- No fixed ports: do not hardcode ports (e.g. 8000). Bind UDP/HTTP to OS-assigned available ports (port=0) by default; invitations/UI carry the actual assigned endpoint. Optional prime-port preference may be applied within the dynamic/private range 49152-65535 (nearest/selected prime) as a preference only - networking simplicity is paramount.
- No pi-sync / no time-server chain: do not require Internet time servers or chained time sync. Use monotonic clocks, local timestamps, and peer heartbeat/ACK semantics for liveness/RTT/timeouts.
- Transparent and simple: work across networks where possible; two connected nodes suffice as a complete connection/mesh with no Internet required.
- Routes, not location: routes are visualized with common chat; physical location is secondary or of no importance.

IDENTITY, CHAT, AND PROOF OF COMMUNICATION
- Peer-to-peer chat showing ping distance (hops/path/latency) and pregenerated/accepted unique nicknames.
- Common chat and invite/group chats (emulated/supported across node types).
- Hail on connect: to prove communication, each newly connected node hails others in common chat with details (nickname, platform/type, node ID, transport/port/endpoint details, peers/context).
- '?' in every client: reveals the detailed complete objective and commands/help (terminal, browser/HTML, Android).
- Blockchain of messages (common chat): common chat is represented as an append-only, hash-linked message chain (index, timestamp, previous hash, hash, author, payload/kind/route metadata) so message history/integrity is visible in the running session.

PLATFORMS AND OUTPUTS
- Simulation: simulate the whole network on several nodes of different type/platform (Android, bash/terminal, web/browser), demonstrating join, hail, chat, ping, groups, reroute, and stability.
- Browser sessions: several browser sessions can open the running session node/page, join, and participate; new nodes smoothly pop in on the Sensei Network Map (responsive animation).
- Terminal: terminal-installable production client/package (meshctl) for invite/join/chat/peers/routes/ping/serve-web/help.
- Android: one Android app/client path exchanging messages with browser/terminal nodes; provide buildable/launch-ready APK project (and compiled APK artifact when toolchain is available).
- Website/HTML: beacon/session HTML page displays common chat and blockchain, serves as joinable session node UI, and is provided as a launchable/downloadable artifact.

OPERATIONAL/RUNTIME GOALS
- Keep simulation running and provide a testing link openable in several browsers to join and see the Blockchain of messages of common chat in the live HTML session node.
- Sensei Network Map: responsive animated network map where nodes/links update and newly joined nodes smoothly pop in.
- Security: signed invitations, authenticated/compression-aware encrypted envelopes, replay/sequence protection, local private keys, no sensitive data leakage in beacon/public views.
- TDD discipline: tests define/validate core behaviors (identity/invites/envelopes/compression/ports/routing/mesh/hail/blockchain/beacon/session APIs); implementation satisfies them.
- Publishing: publish all project materials to GitHub.
"""

OBJECTIVE = FULL_OBJECTIVE

HELP_TEXT = """?
MeshNet complete objective and commands

""" + FULL_OBJECTIVE + """

COMMANDS
  ?                  Show this complete objective/help
  /help              Show this complete objective/help
  /peers             List connected peers and transports
  /routes            Show known routes
  /ping <nickname>   Ping a peer (hops/path/latency)
  /join <room>       Join or create a group chat
  /leave <room>      Leave a group chat
  /invite            Show this node's signed invitation

RUNTIME BEHAVIOR
- On connect, each node sends a HAIL to common chat with details (nickname, platform, node ID, udp_port/endpoint, peers).
- Common chat is an append-only hash-linked Blockchain of messages (index, prev_hash, hash, timestamp, source, payload, route metadata).
- Session UI shows the Sensei Network Map (responsive animation; new nodes pop in smoothly).
- Two connected nodes suffice without Internet; no fixed ports; OS-assigned ports by default.
"""


def show_help() -> str:
    return HELP_TEXT


In [ ]:
%%writefile meshnet_project/meshnet/__init__.py
"""MeshNet package."""
__all__ = [
    "objective", "blockchain", "invite", "envelope", "compression", "port_selection",
    "routing", "node", "mesh", "transport_udp", "web_beacon", "cli",
]


In [ ]:
%%writefile meshnet_project/meshnet/blockchain.py
"""Append-only hash-linked blockchain of common chat messages."""

from __future__ import annotations

import hashlib
import json
import time
from typing import Any, Dict, List, Optional


def _canonical(data: dict) -> str:
    return json.dumps(data, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def compute_hash(block_core: dict) -> str:
    return hashlib.sha256(_canonical(block_core).encode("utf-8")).hexdigest()


class MessageBlockchain:
    def __init__(self) -> None:
        self.chain: List[Dict[str, Any]] = []
        self._create_genesis()

    def _create_genesis(self) -> None:
        core = {
            "index": 0,
            "timestamp": 0.0,
            "prev_hash": "0" * 64,
            "message_id": "genesis",
            "kind": "genesis",
            "source": "system",
            "destination": "*",
            "room": "common",
            "payload": "MeshNet common chat blockchain genesis",
            "path": [],
            "hops": 0,
        }
        block = dict(core)
        block["hash"] = compute_hash(core)
        self.chain.append(block)

    @property
    def head(self) -> Dict[str, Any]:
        return self.chain[-1]

    def append(self, message: Dict[str, Any]) -> Dict[str, Any]:
        prev = self.head
        core = {
            "index": len(self.chain),
            "timestamp": float(message.get("created_at", time.time())),
            "prev_hash": prev["hash"],
            "message_id": message.get("id", ""),
            "kind": message.get("kind", "chat"),
            "source": message.get("source", "?"),
            "destination": message.get("destination", "*"),
            "room": message.get("room", "common"),
            "payload": message.get("payload", ""),
            "path": list(message.get("path", [])),
            "hops": int(message.get("hops", 0) or 0),
        }
        block = dict(core)
        block["hash"] = compute_hash(core)
        self.chain.append(block)
        return block

    def to_list(self) -> List[Dict[str, Any]]:
        return [dict(b) for b in self.chain]

    def verify(self) -> bool:
        if not self.chain:
            return False
        # verify genesis hash and links
        for i, block in enumerate(self.chain):
            core = {k: block[k] for k in [
                "index","timestamp","prev_hash","message_id","kind","source","destination","room","payload","path","hops"
            ]}
            if compute_hash(core) != block.get("hash"):
                return False
            if i == 0:
                if block.get("prev_hash") != "0" * 64:
                    return False
            else:
                if block.get("prev_hash") != self.chain[i-1].get("hash"):
                    return False
        return True


In [ ]:
%%writefile meshnet_project/meshnet/invite.py
"""Signed invitation tokens for slow/contact bootstrap (QR/NFC/BLE/paste)."""

from __future__ import annotations

import base64
import json
import time
import uuid
from typing import Any, Dict, Optional, Tuple

from nacl.encoding import RawEncoder
from nacl.exceptions import BadSignatureError
from nacl.signing import SigningKey, VerifyKey

PREFIX = "MESH1"


def _b64e(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")


def _b64d(data: str) -> bytes:
    pad = "=" * (-len(data) % 4)
    return base64.urlsafe_b64decode(data + pad)


def generate_signing_key() -> SigningKey:
    return SigningKey.generate()


def public_key_b64(signing_key: SigningKey) -> str:
    return _b64e(signing_key.verify_key.encode(encoder=RawEncoder))


def create_invite(
    node_id: str,
    nickname: str,
    host: str,
    port: int,
    signing_key: Optional[SigningKey] = None,
    expires_in_s: Optional[int] = None,
) -> Tuple[str, SigningKey]:
    """Create a signed invite token. Returns (token, signing_key)."""
    signing_key = signing_key or generate_signing_key()
    payload: Dict[str, Any] = {
        "node_id": node_id,
        "nickname": nickname,
        "host": host,
        "port": int(port),
        "public_key": public_key_b64(signing_key),
        "nonce": uuid.uuid4().hex,
        "issued_at": int(time.time()),
    }
    if expires_in_s is not None:
        payload["expires_at"] = payload["issued_at"] + int(expires_in_s)

    payload_bytes = json.dumps(payload, separators=(",", ":"), sort_keys=True).encode("utf-8")
    signature = signing_key.sign(payload_bytes).signature
    token = f"{PREFIX}.{_b64e(payload_bytes)}.{_b64e(signature)}"
    return token, signing_key


def verify_invite(token: str, now: Optional[int] = None) -> Dict[str, Any]:
    """Verify a signed invite token and return its payload."""
    try:
        prefix, payload_b64, sig_b64 = token.split(".", 2)
        if prefix != PREFIX:
            raise ValueError("unsupported invite prefix")
        payload_bytes = _b64d(payload_b64)
        signature = _b64d(sig_b64)
        payload = json.loads(payload_bytes.decode("utf-8"))
    except Exception as exc:
        raise ValueError(f"malformed invite token: {exc}") from exc

    required = {"node_id", "nickname", "host", "port", "public_key", "nonce", "issued_at"}
    if not required.issubset(payload.keys()):
        raise ValueError("invite payload missing required fields")

    try:
        verify_key = VerifyKey(_b64d(payload["public_key"]))
        verify_key.verify(payload_bytes, signature)
    except BadSignatureError as exc:
        raise ValueError("invite signature verification failed") from exc
    except Exception as exc:
        raise ValueError(f"invalid invite public key/signature: {exc}") from exc

    if "expires_at" in payload:
        current = int(time.time()) if now is None else int(now)
        if current > int(payload["expires_at"]):
            raise ValueError("invite token expired")

    payload["port"] = int(payload["port"])
    return payload


In [ ]:
%%writefile meshnet_project/meshnet/envelope.py
"""Encrypted, replay-protected UDP envelopes."""

from __future__ import annotations

import json
import struct
import time
from typing import Any, Dict, Optional, Set, Tuple

from nacl.secret import SecretBox
from nacl.utils import random as nacl_random

MAGIC = b"MN1"
VERSION = 1


def generate_session_key() -> bytes:
    return nacl_random(SecretBox.KEY_SIZE)


def pack(
    sender: str,
    recipient: str,
    stream_id: int,
    sequence: int,
    payload: bytes,
    key: bytes,
    flags: int = 0,
) -> bytes:
    if not isinstance(payload, (bytes, bytearray)):
        raise TypeError("payload must be bytes")
    header = {
        "v": VERSION,
        "sender": sender,
        "recipient": recipient,
        "stream": int(stream_id),
        "seq": int(sequence),
        "flags": int(flags),
        "ts": time.time(),
    }
    header_bytes = json.dumps(header, separators=(",", ":"), sort_keys=True).encode("utf-8")
    box = SecretBox(key)
    nonce = nacl_random(SecretBox.NONCE_SIZE)
    ciphertext = box.encrypt(bytes(payload), nonce).ciphertext
    return MAGIC + struct.pack("!H", len(header_bytes)) + header_bytes + nonce + ciphertext


def unpack(
    raw: bytes,
    key: bytes,
    seen: Optional[Set[Tuple[str, int]]] = None,
) -> Dict[str, Any]:
    if not isinstance(raw, (bytes, bytearray)):
        raise TypeError("raw must be bytes")
    raw = bytes(raw)
    if len(raw) < 5 + SecretBox.NONCE_SIZE:
        raise ValueError("envelope too short")
    if raw[:3] != MAGIC:
        raise ValueError("invalid envelope magic")
    header_len = struct.unpack("!H", raw[3:5])[0]
    start = 5
    end = start + header_len
    if end + SecretBox.NONCE_SIZE > len(raw):
        raise ValueError("truncated envelope")
    try:
        header = json.loads(raw[start:end].decode("utf-8"))
    except Exception as exc:
        raise ValueError(f"invalid envelope header: {exc}") from exc

    if header.get("v") != VERSION:
        raise ValueError("unsupported envelope version")

    nonce = raw[end:end + SecretBox.NONCE_SIZE]
    ciphertext = raw[end + SecretBox.NONCE_SIZE:]
    box = SecretBox(key)
    try:
        plaintext = box.decrypt(ciphertext, nonce)
    except Exception as exc:
        raise ValueError(f"envelope authentication/decryption failed: {exc}") from exc

    replay_key = (str(header.get("sender")), int(header.get("seq", -1)))
    if seen is not None:
        if replay_key in seen:
            raise ValueError("replayed envelope sequence")
        seen.add(replay_key)

    return {
        "version": header["v"],
        "sender": header["sender"],
        "recipient": header["recipient"],
        "stream": header["stream"],
        "sequence": header["seq"],
        "flags": header.get("flags", 0),
        "timestamp": header.get("ts"),
        "payload": bytes(plaintext),
    }


In [ ]:
%%writefile meshnet_project/meshnet/compression.py
"""Compress-before-encrypt payload helpers."""

from __future__ import annotations

import zlib

COMPRESSED_PREFIX = b"\x01Z"
RAW_PREFIX = b"\x00R"
THRESHOLD = 128


def compress_payload(data: bytes, threshold: int = THRESHOLD) -> bytes:
    if not isinstance(data, (bytes, bytearray)):
        raise TypeError("data must be bytes")
    data = bytes(data)
    if len(data) < threshold:
        return RAW_PREFIX + data
    compressed = zlib.compress(data, level=6)
    packed = COMPRESSED_PREFIX + compressed
    raw = RAW_PREFIX + data
    return packed if len(packed) < len(raw) else raw


def decompress_payload(data: bytes) -> bytes:
    if not isinstance(data, (bytes, bytearray)):
        raise TypeError("data must be bytes")
    data = bytes(data)
    if data.startswith(COMPRESSED_PREFIX):
        return zlib.decompress(data[len(COMPRESSED_PREFIX):])
    if data.startswith(RAW_PREFIX):
        return data[len(RAW_PREFIX):]
    raise ValueError("unknown payload compression prefix")


In [ ]:
%%writefile meshnet_project/meshnet/port_selection.py
"""Port selection: OS-assigned by default, optional prime preference (49152-65535)."""

from __future__ import annotations

import random
import socket
from typing import Tuple

DYNAMIC_PORT_MIN = 49152
DYNAMIC_PORT_MAX = 65535


def is_prime(number: int) -> bool:
    n = int(number)
    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0:
        return False
    d = 3
    while d * d <= n:
        if n % d == 0:
            return False
        d += 2
    return True


def nearest_prime_port(preferred: int) -> int:
    preferred = min(max(int(preferred), DYNAMIC_PORT_MIN), DYNAMIC_PORT_MAX)
    lower = preferred
    upper = preferred
    while lower >= DYNAMIC_PORT_MIN or upper <= DYNAMIC_PORT_MAX:
        if lower >= DYNAMIC_PORT_MIN and is_prime(lower):
            return lower
        if upper <= DYNAMIC_PORT_MAX and is_prime(upper):
            return upper
        lower -= 1
        upper += 1
    raise RuntimeError("no prime port available in dynamic range")


def reserve_udp_port(host: str = "::", prefer_prime: bool = False) -> Tuple[socket.socket, int]:
    """Bind a UDP socket and return (socket, actual_port). Default: OS-assigned port=0."""
    family = socket.AF_INET6 if ":" in host else socket.AF_INET
    sock = socket.socket(family, socket.SOCK_DGRAM)
    try:
        if family == socket.AF_INET6:
            try:
                sock.setsockopt(socket.IPPROTO_IPV6, socket.IPV6_V6ONLY, 0)
            except Exception:
                pass
        if prefer_prime:
            start = random.randint(DYNAMIC_PORT_MIN, DYNAMIC_PORT_MAX)
            tried = 0
            candidate = nearest_prime_port(start)
            while tried < 200:
                try:
                    sock.bind((host, candidate))
                    return sock, sock.getsockname()[1]
                except OSError:
                    candidate += 1
                    if candidate > DYNAMIC_PORT_MAX:
                        candidate = DYNAMIC_PORT_MIN
                    candidate = nearest_prime_port(candidate)
                    tried += 1
            sock.bind((host, 0))
        else:
            sock.bind((host, 0))
        return sock, sock.getsockname()[1]
    except Exception:
        sock.close()
        raise


In [ ]:
%%writefile meshnet_project/meshnet/routing.py
"""Topology and shortest-path routing (local link-state computation)."""

from __future__ import annotations

from collections import deque
from typing import Dict, List, Set


class MeshTopology:
    def __init__(self) -> None:
        self.links: Dict[str, Set[str]] = {}

    def add_node(self, node_id: str) -> None:
        self.links.setdefault(node_id, set())

    def add_link(self, a: str, b: str) -> None:
        if a == b:
            return
        self.add_node(a)
        self.add_node(b)
        self.links[a].add(b)
        self.links[b].add(a)

    def remove_link(self, a: str, b: str) -> None:
        if a in self.links:
            self.links[a].discard(b)
        if b in self.links:
            self.links[b].discard(a)

    def remove_node(self, node_id: str) -> None:
        for peer in list(self.links.get(node_id, set())):
            self.links[peer].discard(node_id)
        self.links.pop(node_id, None)

    def neighbors(self, node_id: str) -> Set[str]:
        return set(self.links.get(node_id, set()))

    def route(self, src: str, dst: str) -> List[str]:
        if src not in self.links or dst not in self.links:
            raise ValueError("unknown source or destination")
        if src == dst:
            return [src]
        q = deque([src])
        prev: Dict[str, str | None] = {src: None}
        while q:
            cur = q.popleft()
            for nb in sorted(self.links.get(cur, [])):
                if nb in prev:
                    continue
                prev[nb] = cur
                if nb == dst:
                    path = [dst]
                    while prev[path[-1]] is not None:
                        path.append(prev[path[-1]])  # type: ignore[arg-type]
                    return list(reversed(path))
                q.append(nb)
        raise ValueError(f"no route from {src} to {dst}")


In [ ]:
%%writefile meshnet_project/meshnet/node.py
"""Node model."""

from __future__ import annotations

import random
import string
import uuid
from collections import deque
from dataclasses import dataclass, field
from typing import Deque, Dict, List, Optional, Set

_ADJECTIVES = ("swift", "quiet", "bright", "calm", "bold", "clever", "lunar", "solar", "sensei")
_NOUNS = ("otter", "falcon", "cedar", "river", "comet", "lynx", "ember", "maple", "node")

PLATFORM_CHANNELS = {
    "android": ("wifi", "bluetooth", "proximity", "web"),
    "bash": ("wifi", "bluetooth", "web"),
    "terminal": ("wifi", "bluetooth", "web"),
    "web": ("web",),
}


def generate_nickname(rng: Optional[random.Random] = None) -> str:
    rng = rng or random.Random()
    suffix = "".join(rng.choices(string.ascii_lowercase + string.digits, k=4))
    return f"{rng.choice(_ADJECTIVES)}-{rng.choice(_NOUNS)}-{suffix}"


@dataclass
class Node:
    nickname: str
    platform: str = "terminal"
    node_id: str = field(default_factory=lambda: f"node-{uuid.uuid4().hex[:10]}")
    host: Optional[str] = None
    udp_port: Optional[int] = None
    channels: tuple[str, ...] = field(default_factory=tuple)
    neighbors: Set[str] = field(default_factory=set)
    groups: Set[str] = field(default_factory=set)
    inbox: Deque[dict] = field(default_factory=deque)
    public_log: List[dict] = field(default_factory=list)
    meta: Dict[str, str] = field(default_factory=dict)

    def __post_init__(self) -> None:
        if not self.channels:
            self.channels = PLATFORM_CHANNELS.get(self.platform, ("wifi", "web"))

    def receive(self, message: dict) -> None:
        msg = dict(message)
        self.inbox.append(msg)
        if msg.get("kind") in {"chat", "group_chat", "system"}:
            self.public_log.append(msg)


In [ ]:
%%writefile meshnet_project/meshnet/mesh.py
"""Mesh network with hash-linked common chat blockchain, hails, routes, groups, ping."""

from __future__ import annotations

import random
import threading
import time
import uuid
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Set, Union

from .blockchain import MessageBlockchain
from .node import Node, generate_nickname
from .objective import HELP_TEXT, OBJECTIVE
from .routing import MeshTopology

NodeLike = Union[str, Node]


class NicknameTakenError(ValueError):
    pass


class UnknownNodeError(KeyError):
    pass


@dataclass
class PingResult:
    source: str
    destination: str
    hops: int
    path: List[str]
    latency_ms: float
    reachable: bool = True

    def to_dict(self) -> dict:
        return {
            "source": self.source,
            "destination": self.destination,
            "hops": self.hops,
            "path": list(self.path),
            "latency_ms": round(self.latency_ms, 3),
            "reachable": self.reachable,
        }


class MeshNetwork:
    def __init__(self) -> None:
        self.nodes: Dict[str, Node] = {}
        self.nick_to_id: Dict[str, str] = {}
        self.groups: Dict[str, Set[str]] = {}
        self.topology = MeshTopology()
        self.blockchain = MessageBlockchain()
        self._rng = random.Random(42)
        self._lock = threading.RLock()
        self._seen_block_messages: Set[str] = set()

    def add_node(
        self,
        nickname: Optional[str] = None,
        platform: str = "terminal",
        node_id: Optional[str] = None,
        host: Optional[str] = None,
        udp_port: Optional[int] = None,
    ) -> Node:
        with self._lock:
            nickname = nickname or generate_nickname(self._rng)
            if nickname in self.nick_to_id:
                raise NicknameTakenError(f"nickname already in use: {nickname}")
            node = Node(
                nickname=nickname,
                platform=platform,
                node_id=node_id or f"node-{uuid.uuid4().hex[:10]}",
                host=host,
                udp_port=udp_port,
            )
            self.nodes[node.node_id] = node
            self.nick_to_id[nickname] = node.node_id
            self.topology.add_node(node.node_id)
            return node

    def resolve(self, identifier: NodeLike) -> Node:
        if isinstance(identifier, Node):
            return identifier
        node_id = self.nick_to_id.get(identifier, identifier)
        if node_id not in self.nodes:
            raise UnknownNodeError(identifier)
        return self.nodes[node_id]

    def connect(self, a: NodeLike, b: NodeLike) -> None:
        with self._lock:
            na, nb = self.resolve(a), self.resolve(b)
            if na.node_id == nb.node_id:
                return
            na.neighbors.add(nb.node_id)
            nb.neighbors.add(na.node_id)
            self.topology.add_link(na.node_id, nb.node_id)

    def disconnect(self, a: NodeLike, b: NodeLike) -> None:
        with self._lock:
            na, nb = self.resolve(a), self.resolve(b)
            na.neighbors.discard(nb.node_id)
            nb.neighbors.discard(na.node_id)
            self.topology.remove_link(na.node_id, nb.node_id)

    def remove_node(self, identifier: NodeLike) -> None:
        with self._lock:
            node = self.resolve(identifier)
            nid = node.node_id
            for other in list(node.neighbors):
                if other in self.nodes:
                    self.nodes[other].neighbors.discard(nid)
            self.topology.remove_node(nid)
            for members in list(self.groups.values()):
                members.discard(nid)
            for name in [n for n, m in self.groups.items() if not m]:
                self.groups.pop(name, None)
            self.nick_to_id.pop(node.nickname, None)
            self.nodes.pop(nid, None)

    def show_help(self) -> str:
        return HELP_TEXT

    @property
    def objective(self) -> str:
        return OBJECTIVE

    def _record_chain(self, message: dict) -> None:
        mid = message.get("id")
        if not mid or mid in self._seen_block_messages:
            return
        if message.get("kind") in {"chat", "group_chat", "system"}:
            self.blockchain.append(message)
            self._seen_block_messages.add(mid)

    def _deliver_along_path(self, path_ids: List[str], message: dict) -> dict:
        for nid in path_ids:
            self.nodes[nid].receive(message)
        self._record_chain(message)
        return message

    def broadcast_system_message(self, source: NodeLike, text: str, room: str = "common") -> dict:
        with self._lock:
            src = self.resolve(source)
            message = {
                "id": uuid.uuid4().hex,
                "kind": "system",
                "source": src.nickname,
                "source_id": src.node_id,
                "destination": "*",
                "room": room,
                "payload": text,
                "created_at": time.time(),
                "path": [src.nickname],
                "hops": 0,
            }
            for node in self.nodes.values():
                node.receive(message)
            self._record_chain(message)
            return message

    def hail(self, source: NodeLike) -> dict:
        node = self.resolve(source)
        text = (
            f"HAIL {node.nickname} joined the mesh. "
            f"platform={node.platform}; node_id={node.node_id}; "
            f"udp_port={node.udp_port if node.udp_port is not None else 'unannounced'}; "
            f"peers={len(node.neighbors)}."
        )
        return self.broadcast_system_message(node, text)

    def route_ids(self, src: NodeLike, dst: NodeLike) -> List[str]:
        s = self.resolve(src).node_id
        d = self.resolve(dst).node_id
        return self.topology.route(s, d)

    def route_nicknames(self, src: NodeLike, dst: NodeLike) -> List[str]:
        return [self.nodes[nid].nickname for nid in self.route_ids(src, dst)]

    def render_route(self, src: NodeLike, dst: NodeLike) -> str:
        path = self.route_nicknames(src, dst)
        return " -> ".join(path) if path else "(no route)"

    def send_chat(self, source: NodeLike, destination: NodeLike, text: str) -> dict:
        with self._lock:
            src = self.resolve(source)
            dst = self.resolve(destination)
            path_ids = self.route_ids(src, dst)
            path_names = [self.nodes[nid].nickname for nid in path_ids]
            message = {
                "id": uuid.uuid4().hex,
                "kind": "chat",
                "source": src.nickname,
                "source_id": src.node_id,
                "destination": dst.nickname,
                "destination_id": dst.node_id,
                "room": "common",
                "payload": text,
                "created_at": time.time(),
                "path": path_names,
                "hops": max(0, len(path_ids) - 1),
            }
            return self._deliver_along_path(path_ids, message)

    def broadcast_common_chat(self, source: NodeLike, text: str) -> dict:
        """Send a common-chat message from source to all reachable peers (fan-out best-effort)."""
        with self._lock:
            src = self.resolve(source)
            message = {
                "id": uuid.uuid4().hex,
                "kind": "chat",
                "source": src.nickname,
                "source_id": src.node_id,
                "destination": "*",
                "room": "common",
                "payload": text,
                "created_at": time.time(),
                "path": [src.nickname],
                "hops": 0,
            }
            for node in self.nodes.values():
                node.receive(message)
            self._record_chain(message)
            return message

    def create_group(self, name: str, members: Iterable[NodeLike]) -> Set[str]:
        with self._lock:
            member_ids = {self.resolve(m).node_id for m in members}
            if len(member_ids) < 2:
                raise ValueError("a group needs at least two members")
            self.groups.setdefault(name, set()).update(member_ids)
            for mid in self.groups[name]:
                self.nodes[mid].groups.add(name)
            return set(self.groups[name])

    def join_group(self, name: str, member: NodeLike) -> None:
        with self._lock:
            node = self.resolve(member)
            self.groups.setdefault(name, set()).add(node.node_id)
            node.groups.add(name)

    def leave_group(self, name: str, member: NodeLike) -> None:
        with self._lock:
            node = self.resolve(member)
            if name in self.groups:
                self.groups[name].discard(node.node_id)
                if not self.groups[name]:
                    self.groups.pop(name, None)
            node.groups.discard(name)

    def send_group_chat(self, source: NodeLike, room: str, text: str) -> List[dict]:
        with self._lock:
            src = self.resolve(source)
            if room not in self.groups or src.node_id not in self.groups[room]:
                raise ValueError(f"{src.nickname} is not a member of group {room}")
            delivered: List[dict] = []
            for dst_id in sorted(self.groups[room]):
                if dst_id == src.node_id:
                    continue
                try:
                    path_ids = self.route_ids(src, dst_id)
                except Exception:
                    continue
                path_names = [self.nodes[nid].nickname for nid in path_ids]
                message = {
                    "id": uuid.uuid4().hex,
                    "kind": "group_chat",
                    "source": src.nickname,
                    "source_id": src.node_id,
                    "destination": self.nodes[dst_id].nickname,
                    "destination_id": dst_id,
                    "room": room,
                    "payload": text,
                    "created_at": time.time(),
                    "path": path_names,
                    "hops": max(0, len(path_ids) - 1),
                }
                self._deliver_along_path(path_ids, message)
                delivered.append(message)
            return delivered

    def ping(self, source: NodeLike, destination: NodeLike) -> PingResult:
        src = self.resolve(source)
        dst = self.resolve(destination)
        path_ids = self.route_ids(src, dst)
        path_names = [self.nodes[nid].nickname for nid in path_ids]
        hops = max(0, len(path_ids) - 1)
        base = 3.0
        per_hop_rtt = 8.0
        jitter = (hash((src.node_id, dst.node_id, hops, len(self.nodes))) % 13) * 0.25
        latency_ms = base + hops * per_hop_rtt + jitter
        return PingResult(source=src.nickname, destination=dst.nickname, hops=hops, path=path_names, latency_ms=latency_ms)

    def snapshot(self) -> dict:
        with self._lock:
            nodes = [
                {
                    "id": n.node_id,
                    "nickname": n.nickname,
                    "platform": n.platform,
                    "host": n.host,
                    "udp_port": n.udp_port,
                    "channels": list(n.channels),
                    "peers": [self.nodes[p].nickname for p in sorted(n.neighbors) if p in self.nodes],
                    "groups": sorted(n.groups),
                }
                for n in self.nodes.values()
            ]
            links = []
            seen = set()
            for n in self.nodes.values():
                for p in n.neighbors:
                    if p in self.nodes:
                        a, b = sorted([n.nickname, self.nodes[p].nickname])
                        key = (a, b)
                        if key in seen:
                            continue
                        seen.add(key)
                        links.append({"source": a, "target": b})
            groups = {
                name: sorted(self.nodes[mid].nickname for mid in members if mid in self.nodes)
                for name, members in self.groups.items()
            }
            return {
                "nodes": nodes,
                "links": links,
                "groups": groups,
                "chain": self.blockchain.to_list(),
                "chain_head": self.blockchain.head["hash"],
                "chain_valid": self.blockchain.verify(),
            }


In [ ]:
%%writefile meshnet_project/meshnet/transport_udp.py
"""Asyncio UDP transport (simple encrypted envelope exchange)."""

from __future__ import annotations

import asyncio
from typing import Callable, Optional, Tuple

from .envelope import pack, unpack
from .port_selection import reserve_udp_port


class MeshDatagramProtocol(asyncio.DatagramProtocol):
    def __init__(self, on_message: Callable[[dict, Tuple[str, int]], None], key: bytes):
        self.on_message = on_message
        self.key = key
        self.transport: Optional[asyncio.transports.DatagramTransport] = None
        self.seen = set()

    def connection_made(self, transport):
        self.transport = transport

    def datagram_received(self, data, addr):
        try:
            msg = unpack(data, self.key, seen=self.seen)
            self.on_message(msg, addr)
        except Exception:
            return


async def bind_udp_endpoint(
    host: str = "::",
    prefer_prime: bool = False,
    key: Optional[bytes] = None,
    on_message: Optional[Callable[[dict, Tuple[str, int]], None]] = None,
):
    if key is None:
        raise ValueError("session key required")
    on_message = on_message or (lambda msg, addr: None)
    loop = asyncio.get_running_loop()
    sock, port = reserve_udp_port(host=host, prefer_prime=prefer_prime)
    transport, protocol = await loop.create_datagram_endpoint(
        lambda: MeshDatagramProtocol(on_message, key),
        sock=sock,
    )
    return transport, protocol, port


def send_envelope(transport, addr, sender, recipient, stream_id, sequence, payload: bytes, key: bytes, flags: int = 0):
    raw = pack(sender, recipient, stream_id, sequence, payload, key, flags=flags)
    transport.sendto(raw, addr)


In [ ]:
%%writefile meshnet_project/meshnet/web_beacon.py
"""Live session beacon: multi-browser join, Blockchain of messages, Sensei Network Map."""

from __future__ import annotations

import json
import threading
import traceback
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from typing import Any
from urllib.parse import urlparse

from .objective import HELP_TEXT


class BeaconHandle:
    def __init__(self, server: ThreadingHTTPServer, thread: threading.Thread, url: str, port: int):
        self.server = server
        self.thread = thread
        self.url = url
        self.port = port

    def stop(self) -> None:
        try:
            self.server.shutdown()
            self.server.server_close()
        except Exception:
            pass
        if self.thread.is_alive():
            self.thread.join(timeout=2.0)


def _json_response(handler, obj: Any, status: int = 200):
    body = json.dumps(obj, ensure_ascii=False).encode("utf-8")
    handler.send_response(status)
    handler.send_header("Content-Type", "application/json; charset=utf-8")
    handler.send_header("Content-Length", str(len(body)))
    handler.send_header("Cache-Control", "no-store")
    handler.end_headers()
    handler.wfile.write(body)


def _read_json(handler) -> dict:
    try:
        length = int(handler.headers.get("Content-Length", "0") or "0")
    except Exception:
        length = 0
    raw = handler.rfile.read(length) if length > 0 else b"{}"
    try:
        return json.loads(raw.decode("utf-8") or "{}")
    except Exception:
        return {}


HTML_PAGE = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>MeshNet Live Session — Blockchain & Sensei Network Map</title>
<style>
:root { --bg:#0b1020; --card:#141b33; --muted:#93a4c7; --text:#eaf0ff; --accent:#6ee7ff; --accent2:#a78bfa; --ok:#34d399; --warn:#fbbf24; }
*{box-sizing:border-box} body{margin:0;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;background:linear-gradient(180deg,#070b17,#0b1020);color:var(--text)}
header{padding:14px 18px;border-bottom:1px solid #243055;display:flex;gap:12px;align-items:center;justify-content:space-between;flex-wrap:wrap}
h1{font-size:18px;margin:0} .sub{color:var(--muted);font-size:13px}
.container{padding:14px;display:grid;grid-template-columns:1.2fr 1fr;gap:14px}
.card{background:rgba(20,27,51,.9);border:1px solid #243055;border-radius:14px;padding:12px;min-height:120px}
.card h2{margin:0 0 8px 0;font-size:15px}
@media (max-width: 980px){.container{grid-template-columns:1fr}}
#mapWrap{width:100%;height:420px;position:relative;border-radius:10px;background:radial-gradient(circle at 30% 20%,#111a33,#0a1022);overflow:hidden;border:1px solid #223055}
svg{width:100%;height:100%;display:block}
.node circle{fill:url(#nodeGrad);stroke:#bfefff;stroke-width:1.5;filter:drop-shadow(0 0 6px rgba(110,231,255,.35));transform-origin:center;transform-box:fill-box}
.node text{fill:#eaf0ff;font-size:11px;pointer-events:none;text-anchor:middle}
.node .meta{fill:#93a4c7;font-size:9px}
.node.pop circle{animation:popin .55s cubic-bezier(.2,.9,.2,1.2)}
@keyframes popin{0%{transform:scale(0.1);opacity:0}60%{transform:scale(1.18);opacity:1}100%{transform:scale(1);opacity:1}}
.link{stroke:rgba(167,139,250,.55);stroke-width:2;stroke-linecap:round;opacity:.9}
.link.new{animation:linkin .6s ease}
@keyframes linkin{from{opacity:0}to{opacity:.9}}
.controls{display:flex;gap:8px;flex-wrap:wrap;align-items:center}
button,input{border-radius:10px;border:1px solid #2a3a66;background:#0e1630;color:var(--text);padding:8px 10px}
button{cursor:pointer;background:linear-gradient(180deg,#18244a,#101938)} button:hover{border-color:#4b6cb7}
input{min-width:180px}
.pill{display:inline-block;padding:2px 8px;border-radius:999px;background:#102044;border:1px solid #2b3f70;color:#cfe3ff;font-size:12px;margin-left:6px}
#chat{max-height:420px;overflow:auto;display:flex;flex-direction:column;gap:8px;padding-right:4px}
.block{border:1px solid #2a3a66;border-radius:10px;padding:8px;background:#0e1630}
.block .top{display:flex;justify-content:space-between;gap:8px;flex-wrap:wrap;font-size:12px;color:var(--muted)}
.block .payload{margin-top:4px;white-space:pre-wrap;word-break:break-word}
.hash{font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11px;color:#a5b4fc}
.muted{color:var(--muted)} .ok{color:var(--ok)} .warn{color:var(--warn)}
#helpPanel{display:none;position:fixed;inset:5% 6%;background:#0b1124;border:1px solid #334155;border-radius:14px;padding:14px;z-index:50;overflow:auto;box-shadow:0 20px 60px rgba(0,0,0,.6)}
#helpPanel pre{white-space:pre-wrap;color:#dbe7ff;font-family:ui-monospace,Menlo,monospace;font-size:12px}
#overlay{display:none;position:fixed;inset:0;background:rgba(0,0,0,.5);z-index:40}
.row{display:flex;gap:8px;flex-wrap:wrap;margin-top:8px}
.small{font-size:12px}
footer{padding:8px 14px;color:var(--muted);font-size:12px}
</style>
</head>
<body>
<header>
  <div>
    <h1>MeshNet Live Session <span class="pill">Blockchain of messages</span> <span class="pill">Sensei Network Map</span></h1>
    <div class="sub">No fixed ports · OS-assigned endpoint · 2 nodes suffice · No Internet required for mesh core · Live multi-browser join</div>
  </div>
  <div class="controls">
    <button id="helpBtn">?</button>
    <button id="joinBtn">Join as browser node</button>
    <span id="me" class="muted">not joined</span>
  </div>
</header>
<div class="container">
  <section class="card">
    <h2>Sensei Network Map <span id="nodeCount" class="pill">0 nodes</span> <span id="linkCount" class="pill">0 links</span></h2>
    <div id="mapWrap">
      <svg id="map" viewBox="0 0 900 420" preserveAspectRatio="xMidYMid meet">
        <defs>
          <radialGradient id="nodeGrad" cx="35%" cy="30%" r="70%">
            <stop offset="0%" stop-color="#9ff3ff"/><stop offset="100%" stop-color="#4f46e5"/>
          </radialGradient>
        </defs>
        <g id="links"></g>
        <g id="nodes"></g>
      </svg>
    </div>
    <div class="row small muted">
      <span>New nodes smoothly pop in when they join. Routes/links update responsively.</span>
      <span id="status"></span>
    </div>
    <div class="row">
      <input id="chatInput" placeholder="Type message and press Enter (join as browser node to send as yourself)" />
      <button id="sendBtn">Send</button>
    </div>
  </section>
  <section class="card">
    <h2>Blockchain of messages (common chat) <span id="chainInfo" class="pill">verifying…</span></h2>
    <div id="chat"></div>
  </section>
</div>
<div id="overlay"></div>
<div id="helpPanel"><div style="display:flex;justify-content:space-between;align-items:center"><h2 style="margin:0">? Complete Objective &amp; Commands</h2><button id="closeHelp">Close</button></div><pre id="helpText"></pre></div>
<footer>Open this same testing link in several browsers, click "Join as browser node", and watch nodes pop into the Sensei Network Map while the message blockchain grows.</footer>
<script>
const state = {data:null, knownNodes:new Set(), knownLinks:new Set(), joinedId:null};
const el = id => document.getElementById(id);
async function api(path, opts){
  opts = opts || {};
  const base = {headers:{'Content-Type':'application/json'}};
  const r = await fetch(path, Object.assign(base, opts));
  return r.json();
}
function esc(s){return (s==null?'':String(s)).replace(/[&<>"']/g,function(c){return {"&":"&amp;","<":"&lt;",">":"&gt;","\"":"&quot;","'":"&#39;"}[c];});}
function shortHash(h){return h? (h.slice(0,10)+'…'+h.slice(-8)) : '';}
function hashCode(s){s=String(s||''); var h=0; for(var i=0;i<s.length;i++){h=((h<<5)-h)+s.charCodeAt(i); h|=0;} return Math.abs(h);}
function layout(nodes){
  var w=900,h=420,cx=w/2,cy=h/2; var n=nodes.length||1; var R=Math.min(w,h)*0.33 + Math.min(40,n*2);
  var pos={};
  nodes.forEach(function(node,i){
    var a=(Math.PI*2*i)/n - Math.PI/2;
    var jitter=((hashCode(node.nickname)%7)-3)*6;
    pos[node.nickname]={x:cx + (R+jitter)*Math.cos(a), y:cy + (R+jitter)*Math.sin(a)};
  });
  return pos;
}
function renderMap(data){
  var nodes = data.nodes||[]; var links=data.links||[];
  var pos = layout(nodes);
  var linksG = el('links'); var nodesG = el('nodes');
  linksG.innerHTML = links.map(function(l){
    var a=pos[l.source], b=pos[l.target]; if(!a||!b) return '';
    var k=l.source+'--'+l.target; var cls = state.knownLinks.has(k)?'link':'link new'; state.knownLinks.add(k);
    return '<line class="'+cls+'" x1="'+a.x+'" y1="'+a.y+'" x2="'+b.x+'" y2="'+b.y+'" />';
  }).join('');
  nodesG.innerHTML = nodes.map(function(n){
    var p=pos[n.nickname]||{x:450,y:210}; var isNew=!state.knownNodes.has(n.nickname); state.knownNodes.add(n.nickname);
    var cls = 'node' + (isNew?' pop':'');
    var label = esc(n.nickname);
    var meta = esc(n.platform + ' · udp:' + (n.udp_port!=null?n.udp_port:'n/a') + ' · peers:' + ((n.peers||[]).length));
    var shortLabel = label.length>14?label.slice(0,13)+'…':label;
    var shortMeta = meta.length>30?meta.slice(0,29)+'…':meta;
    return '<g class="'+cls+'" transform="translate('+p.x+','+p.y+')">' +
      '<circle r="22"></circle>' +
      '<text y="4">'+shortLabel+'</text>' +
      '<text class="meta" y="38">'+shortMeta+'</text>' +
    '</g>';
  }).join('');
  el('nodeCount').textContent = nodes.length + ' nodes';
  el('linkCount').textContent = links.length + ' links';
}
function renderChain(data){
  var chain = (data.chain||[]).slice().reverse();
  var valid = !!data.chain_valid;
  el('chainInfo').textContent = (valid?'valid':'INVALID') + ' · ' + (data.chain?data.chain.length:0) + ' blocks';
  el('chainInfo').className = 'pill ' + (valid?'ok':'warn');
  el('chat').innerHTML = chain.map(function(b){
    var route = (b.path&&b.path.length)?(' · route: ' + esc(b.path.join(' -> ')) + ' (hops ' + (b.hops||0) + ')') : '';
    var t = b.timestamp ? new Date(b.timestamp*1000).toLocaleTimeString() : '';
    return '<div class="block">' +
      '<div class="top"><span>#'+b.index+' · <strong>'+esc(b.source)+'</strong> · '+esc(b.kind)+route+'</span><span>'+esc(t)+'</span></div>' +
      '<div class="payload">'+esc(b.payload)+'</div>' +
      '<div class="hash">hash: '+shortHash(b.hash)+' · prev: '+shortHash(b.prev_hash)+'</div>' +
    '</div>';
  }).join('');
}
async function refresh(){
  try{
    var data = await api('/api/state');
    state.data = data;
    renderMap(data); renderChain(data);
    if(data.help && !el('helpText').textContent) el('helpText').textContent = data.help;
    el('status').textContent = ' · head ' + shortHash(data.chain_head) + ' · ' + new Date().toLocaleTimeString();
  }catch(e){ el('status').textContent = ' · reconnecting…'; }
}
async function join(){
  try{
    var res = await api('/api/join', {method:'POST', body: JSON.stringify({platform:'web'})});
    if(res && res.node){ state.joinedId = res.node.nickname; el('me').textContent = 'joined as ' + res.node.nickname; el('me').className='ok'; }
    await refresh();
  }catch(e){ console.error(e); }
}
async function send(){
  var text = el('chatInput').value.trim(); if(!text) return;
  try{
    if(text === '?' || text === '/help'){ el('helpPanel').style.display='block'; el('overlay').style.display='block'; el('chatInput').value=''; return; }
    await api('/api/chat', {method:'POST', body: JSON.stringify({from: state.joinedId, text: text})});
    el('chatInput').value='';
    await refresh();
  }catch(e){ console.error(e); }
}
el('joinBtn').onclick = join; el('sendBtn').onclick = send;
el('chatInput').addEventListener('keydown', function(e){ if(e.key==='Enter') send(); });
el('helpBtn').onclick = function(){ el('helpPanel').style.display='block'; el('overlay').style.display='block'; };
el('closeHelp').onclick = function(){ el('helpPanel').style.display='none'; el('overlay').style.display='none'; };
el('overlay').onclick = function(){ el('helpPanel').style.display='none'; el('overlay').style.display='none'; };
refresh(); setInterval(refresh, 1200);
</script>
</body>
</html>
"""


def start_beacon(network, host: str = "127.0.0.1", port: int = 0) -> BeaconHandle:
    """Start live session beacon on OS-assigned port by default (port=0)."""
    state_lock = threading.RLock()

    def ensure_host():
        with state_lock:
            if not network.nodes:
                host_node = network.add_node(nickname="session-host", platform="terminal", udp_port=None)
                network.hail(host_node)

    ensure_host()

    class Handler(BaseHTTPRequestHandler):
        def _send_html(self, body: str, status: int = 200):
            data = body.encode("utf-8")
            self.send_response(status)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.send_header("Content-Length", str(len(data)))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            self.wfile.write(data)

        def do_GET(self):  # noqa: N802
            try:
                parsed = urlparse(self.path)
                if parsed.path in ("/", "/index.html"):
                    return self._send_html(HTML_PAGE)
                if parsed.path in ("/api/state", "/state.json"):
                    snap = network.snapshot()
                    snap["help"] = HELP_TEXT
                    return _json_response(self, snap)
                if parsed.path == "/api/help":
                    return _json_response(self, {"help": HELP_TEXT})
                self.send_error(404)
            except Exception:
                traceback.print_exc()
                try:
                    self.send_error(500)
                except Exception:
                    pass

        def do_POST(self):  # noqa: N802
            try:
                parsed = urlparse(self.path)
                payload = _read_json(self)
                if parsed.path == "/api/join":
                    with state_lock:
                        ensure_host()
                        platform = str(payload.get("platform") or "web")
                        nickname = payload.get("nickname")
                        node = network.add_node(nickname=nickname, platform=platform, udp_port=None, host="browser")
                        candidates = [n for n in network.nodes.values() if n.node_id != node.node_id]
                        if candidates:
                            target = None
                            for c in candidates:
                                if c.nickname == "session-host":
                                    target = c
                                    break
                            if target is None:
                                target = candidates[0]
                            network.connect(node, target)
                        network.hail(node)
                        snap = network.snapshot()
                        snap["help"] = HELP_TEXT
                    return _json_response(self, {"ok": True, "node": {
                        "id": node.node_id, "nickname": node.nickname, "platform": node.platform,
                        "udp_port": node.udp_port,
                        "peers": [network.nodes[p].nickname for p in node.neighbors if p in network.nodes]
                    }, "state": snap})

                if parsed.path == "/api/chat":
                    text = str(payload.get("text", "")).strip()
                    frm = payload.get("from")
                    if not text:
                        return _json_response(self, {"ok": False, "error": "empty text"}, status=400)
                    with state_lock:
                        if not frm or (frm not in network.nick_to_id and frm not in network.nodes):
                            frm = "session-host" if "session-host" in network.nick_to_id else (next(iter(network.nick_to_id), None))
                        if not frm:
                            return _json_response(self, {"ok": False, "error": "unknown sender; join first"}, status=400)
                        if text in ("?", "/help"):
                            msg = {"kind": "help", "payload": HELP_TEXT}
                        else:
                            msg = network.broadcast_common_chat(frm, text)
                        snap = network.snapshot()
                        snap["help"] = HELP_TEXT
                    return _json_response(self, {"ok": True, "message": msg, "state": snap})

                self.send_error(404)
            except Exception as e:
                traceback.print_exc()
                return _json_response(self, {"ok": False, "error": str(e)}, status=500)

        def log_message(self, fmt, *args):
            return

    try:
        server = ThreadingHTTPServer((host, port), Handler)
    except Exception:
        fallback = "127.0.0.1"
        server = ThreadingHTTPServer((fallback, 0), Handler)
        host = fallback

    actual_host, actual_port = server.server_address[0], server.server_address[1]
    if ":" in str(actual_host):
        url = f"http://[{actual_host}]:{actual_port}/"
    else:
        url = f"http://{actual_host}:{actual_port}/"

    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    return BeaconHandle(server=server, thread=thread, url=url, port=actual_port)


In [ ]:
%%writefile meshnet_project/meshnet/cli.py
"""Terminal CLI (meshctl) - protocol-aligned production scaffold."""

from __future__ import annotations

import argparse
import json

from .invite import create_invite
from .mesh import MeshNetwork
from .objective import HELP_TEXT
from .port_selection import reserve_udp_port
from .web_beacon import start_beacon


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(prog="meshctl", description="MeshNet terminal client")
    sub = parser.add_subparsers(dest="cmd")

    sub.add_parser("help", help="Show complete objective and commands")

    p_inv = sub.add_parser("invite", help="Generate a signed invitation (allocates OS UDP port)")
    p_inv.add_argument("--nickname", default="terminal-node")
    p_inv.add_argument("--host", default="::1")
    p_inv.add_argument("--prefer-prime", action="store_true")

    p_serve = sub.add_parser("serve-web", help="Start live session beacon on OS-assigned port")
    p_serve.add_argument("--host", default="127.0.0.1")

    sub.add_parser("demo", help="Run local demo")

    args = parser.parse_args(argv)

    if args.cmd in (None, "help"):
        print(HELP_TEXT)
        return 0

    if args.cmd == "invite":
        sock, port = reserve_udp_port(host="::", prefer_prime=args.prefer_prime)
        sock.close()
        token, _key = create_invite(node_id="node-cli", nickname=args.nickname, host=args.host, port=port)
        print(json.dumps({"nickname": args.nickname, "host": args.host, "udp_port": port, "token": token}, indent=2))
        return 0

    if args.cmd == "serve-web":
        net = MeshNetwork()
        beacon = start_beacon(net, host=args.host, port=0)
        print(f"Session beacon: {beacon.url}")
        print("Open in browsers to join, view Blockchain of messages and Sensei Network Map. Ctrl+C to stop.")
        try:
            import time
            while True:
                time.sleep(1)
        except KeyboardInterrupt:
            beacon.stop()
        return 0

    if args.cmd == "demo":
        net = MeshNetwork()
        a = net.add_node("android-node", platform="android", udp_port=52417)
        b = net.add_node("terminal-node", platform="terminal", udp_port=53128)
        net.connect(a, b)
        net.hail(a); net.hail(b)
        print(net.render_route(a, b))
        print(json.dumps(net.ping(a, b).to_dict(), indent=2))
        print("chain_valid:", net.blockchain.verify(), "blocks:", len(net.blockchain.chain))
        return 0

    parser.print_help()
    return 1


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile meshnet_project/pyproject.toml
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "meshnet"
version = "0.2.0"
description = "MeshNet: resilient P2P mesh with invites, encrypted UDP, routing, blockchain chat, live session beacon, CLI"
readme = "README.md"
requires-python = ">=3.10"
dependencies = [
  "pynacl>=1.5",
  "websockets>=12",
]

[project.optional-dependencies]
dev = ["pytest>=8", "pytest-asyncio>=0.23", "build>=1.0.0"]

[project.scripts]
meshctl = "meshnet.cli:main"

[tool.setuptools]
packages = ["meshnet"]

[tool.pytest.ini_options]
addopts = "-q"
asyncio_mode = "auto"


In [ ]:
%%writefile meshnet_project/README.md
# MeshNet (complete project)

Foregrounds the full extracted project objective: TDD, Hamachi/Skype-like router-node-each mesh, Wi-Fi/BT/near-neighbors channel model, IPv6-first 1-UDP-port compression-encrypted peer links, slow bootstrap invites (QR/NFC/BLE/paste), no fixed ports (OS-assigned, optional prime preference 49152-65535), no pi-sync/time-server chain, 2 nodes suffice without Internet, `?` in every client, hail on connect, routes visualized, and a live session beacon showing **Blockchain of messages (common chat)** and **Sensei Network Map** (responsive animated pop-in for new nodes).

## Install (terminal-installable)
```bash
python -m venv .venv
source .venv/bin/activate  # Windows: .venv\Scripts\activate
pip install -e .[dev]
```

## CLI
```bash
meshctl help
meshctl invite --nickname terminal-node --host ::1
meshctl serve-web
meshctl demo
```

## Tests
```bash
pytest
```

## Live session
`meshctl serve-web` starts an OS-assigned session beacon. Open the printed URL in several browsers, click **Join as browser node**, and observe the Blockchain of messages and Sensei Network Map updates.

## Outputs
- Launchable HTML session client: `dist/meshnet.html` (or served by beacon)
- Terminal installable package/project: `meshnet_project/` (+ distributable archive)
- Android APK build project: `android_apk_project/` (+ archive); build APK with Android/Gradle toolchain

## Android APK build
```bash
cd android_apk_project
./gradlew assembleDebug
./gradlew assembleRelease
```
(Requires Android SDK/JDK/Gradle environment.)


## Tests (TDD validation)
Covers objective/help, invites, envelopes, compression, port selection, routing, mesh (hails/chat/groups/ping/reroute), hash-linked blockchain, UDP transport loopback, and live session beacon APIs.


In [ ]:
%%writefile meshnet_project/tests/test_objective.py
from meshnet.objective import HELP_TEXT, FULL_OBJECTIVE, show_help
from meshnet.mesh import MeshNetwork


def test_complete_objective_present():
    assert "COMPLETE PROJECT OBJECTIVE" in FULL_OBJECTIVE
    assert "Blockchain of messages" in FULL_OBJECTIVE
    assert "Sensei Network Map" in FULL_OBJECTIVE
    assert "Runtime -> Run all" in FULL_OBJECTIVE or "Runtime" in FULL_OBJECTIVE
    assert "No fixed ports" in FULL_OBJECTIVE


def test_help_contains_objective_and_commands():
    help_text = show_help()
    assert "COMPLETE PROJECT OBJECTIVE" in help_text
    assert "Blockchain of messages" in help_text
    assert "Sensei Network Map" in help_text
    assert "/peers" in HELP_TEXT
    assert "/routes" in HELP_TEXT
    assert "/ping" in HELP_TEXT


def test_network_show_help_matches_shared_help():
    net = MeshNetwork()
    assert net.show_help() == HELP_TEXT


In [ ]:
%%writefile meshnet_project/tests/test_blockchain.py
from meshnet.blockchain import MessageBlockchain


def test_genesis_and_append_and_verify():
    bc = MessageBlockchain()
    assert len(bc.chain) == 1
    assert bc.chain[0]["index"] == 0
    assert bc.chain[0]["prev_hash"] == "0" * 64
    assert bc.verify()

    b1 = bc.append({"id": "m1", "kind": "chat", "source": "a", "payload": "hi", "created_at": 1.0, "path": ["a", "b"], "hops": 1})
    b2 = bc.append({"id": "m2", "kind": "system", "source": "b", "payload": "HAIL", "created_at": 2.0, "path": ["b"], "hops": 0})
    assert b1["index"] == 1 and b2["index"] == 2
    assert b1["prev_hash"] == bc.chain[0]["hash"]
    assert b2["prev_hash"] == b1["hash"]
    assert b2["hash"] == bc.head["hash"]
    assert bc.verify()


def test_tamper_detected():
    bc = MessageBlockchain()
    bc.append({"id": "m1", "kind": "chat", "source": "a", "payload": "hi", "created_at": 1.0, "path": ["a"], "hops": 0})
    assert bc.verify()
    bc.chain[1]["payload"] = "tampered"
    assert not bc.verify()


In [ ]:
%%writefile meshnet_project/tests/test_invite.py
import pytest
from meshnet.invite import create_invite, verify_invite


def test_invite_roundtrip():
    token, _key = create_invite("node-a", "swift-otter", "2001:db8::1", 4242)
    info = verify_invite(token)
    assert info["node_id"] == "node-a"
    assert info["nickname"] == "swift-otter"
    assert info["host"] == "2001:db8::1"
    assert info["port"] == 4242
    assert "public_key" in info


def test_tampered_invite_rejected():
    token, _key = create_invite("node-a", "swift-otter", "2001:db8::1", 4242)
    tampered = token[:-1] + ("A" if token[-1] != "A" else "B")
    with pytest.raises(ValueError):
        verify_invite(tampered)


def test_expired_invite_rejected():
    token, _key = create_invite("node-a", "swift-otter", "2001:db8::1", 4242, expires_in_s=10)
    with pytest.raises(ValueError):
        verify_invite(token, now=10**12)


In [ ]:
%%writefile meshnet_project/tests/test_envelope.py
import pytest
from meshnet.envelope import generate_session_key, pack, unpack


def test_envelope_roundtrip():
    key = generate_session_key()
    raw = pack("node-a", "node-b", 1, 0, b"hello mesh", key)
    msg = unpack(raw, key)
    assert msg["sender"] == "node-a"
    assert msg["recipient"] == "node-b"
    assert msg["payload"] == b"hello mesh"
    assert msg["sequence"] == 0


def test_replay_rejected():
    key = generate_session_key()
    raw = pack("node-a", "node-b", 1, 7, b"hello", key)
    seen = set()
    unpack(raw, key, seen=seen)
    with pytest.raises(ValueError):
        unpack(raw, key, seen=seen)


def test_tampered_ciphertext_rejected():
    key = generate_session_key()
    raw = bytearray(pack("node-a", "node-b", 1, 1, b"secret", key))
    raw[-1] ^= 0x01
    with pytest.raises(ValueError):
        unpack(bytes(raw), key)


In [ ]:
%%writefile meshnet_project/tests/test_compression.py
from meshnet.compression import compress_payload, decompress_payload


def test_small_payload_roundtrip():
    data = b"hi"
    packed = compress_payload(data)
    assert decompress_payload(packed) == data


def test_large_payload_roundtrip():
    data = b"hello-mesh-" * 200
    packed = compress_payload(data)
    assert decompress_payload(packed) == data


In [ ]:
%%writefile meshnet_project/tests/test_port_selection.py
from meshnet.port_selection import is_prime, nearest_prime_port, reserve_udp_port


def test_prime_detection():
    assert is_prime(2)
    assert is_prime(3)
    assert is_prime(49157)
    assert not is_prime(1)
    assert not is_prime(4)
    assert not is_prime(49152)


def test_nearest_prime_port_in_dynamic_range():
    p = nearest_prime_port(50000)
    assert is_prime(p)
    assert 49152 <= p <= 65535


def test_os_assigned_udp_port_default():
    sock, port = reserve_udp_port(host="127.0.0.1", prefer_prime=False)
    try:
        assert isinstance(port, int)
        assert 1 <= port <= 65535
    finally:
        sock.close()


In [ ]:
%%writefile meshnet_project/tests/test_routing.py
import pytest
from meshnet.routing import MeshTopology


def test_multihop_route():
    t = MeshTopology()
    t.add_link("a", "b")
    t.add_link("b", "c")
    assert t.route("a", "c") == ["a", "b", "c"]


def test_failure_reroutes():
    t = MeshTopology()
    t.add_link("a", "b")
    t.add_link("b", "c")
    t.add_link("a", "d")
    t.add_link("d", "c")
    assert t.route("a", "c") == ["a", "b", "c"]
    t.remove_node("b")
    assert t.route("a", "c") == ["a", "d", "c"]


def test_no_route_raises():
    t = MeshTopology()
    t.add_node("a")
    t.add_node("z")
    with pytest.raises(ValueError):
        t.route("a", "z")


In [ ]:
%%writefile meshnet_project/tests/test_mesh.py
import pytest
from meshnet.mesh import MeshNetwork, NicknameTakenError
from meshnet.objective import HELP_TEXT


def test_unique_nicknames_and_resolve():
    net = MeshNetwork()
    a = net.add_node("fixed-node", platform="android")
    b = net.add_node(platform="terminal")
    assert a.nickname == "fixed-node"
    assert b.nickname != a.nickname
    assert net.resolve("fixed-node").node_id == a.node_id
    with pytest.raises(NicknameTakenError):
        net.add_node("fixed-node")


def test_direct_chat_ping_route_and_blockchain():
    net = MeshNetwork()
    a = net.add_node("alice", platform="android", udp_port=50001)
    b = net.add_node("bob", platform="terminal", udp_port=50002)
    net.connect(a, b)
    msg = net.send_chat("alice", "bob", "hello")
    assert msg["hops"] == 1
    assert msg["path"] == ["alice", "bob"]
    assert any(m["payload"] == "hello" and m["kind"] == "chat" for m in b.public_log)
    ping = net.ping("alice", "bob")
    assert ping.hops == 1 and ping.path == ["alice", "bob"] and ping.latency_ms > 0
    assert net.render_route("alice", "bob") == "alice -> bob"
    assert net.blockchain.verify()
    assert len(net.blockchain.chain) >= 2  # genesis + chat


def test_multihop_group_reroute_hail_and_chain():
    net = MeshNetwork()
    a = net.add_node("android-node", platform="android", udp_port=52417)
    b = net.add_node("terminal-node", platform="terminal", udp_port=53128)
    c = net.add_node("browser-node", platform="web")
    d = net.add_node("alt-node", platform="bash")
    net.connect(a, b); net.connect(b, c)
    net.connect(a, d); net.connect(d, c)

    assert net.route_nicknames(a, c) == ["android-node", "terminal-node", "browser-node"]
    net.create_group("team", [a, b, c])
    delivered = net.send_group_chat("android-node", "team", "status")
    assert len(delivered) == 2

    net.remove_node(b)
    assert net.route_nicknames(a, c) == ["android-node", "alt-node", "browser-node"]
    assert net.ping(a, c).hops == 2

    hail = net.hail(a)
    assert hail["kind"] == "system"
    assert "HAIL android-node joined the mesh" in hail["payload"]
    assert "udp_port=52417" in hail["payload"]
    assert any(m["payload"] == hail["payload"] for m in c.public_log)

    assert net.blockchain.verify()
    assert len(net.blockchain.chain) >= 4  # genesis + group msgs + hail


def test_help_available():
    net = MeshNetwork()
    assert net.show_help() == HELP_TEXT


In [ ]:
%%writefile meshnet_project/tests/test_transport.py
import asyncio
import pytest

from meshnet.envelope import generate_session_key
from meshnet.transport_udp import bind_udp_endpoint, send_envelope


@pytest.mark.asyncio
async def test_udp_loopback_envelope_exchange():
    key = generate_session_key()
    received = []

    def on_msg(msg, addr):
        received.append(msg)

    transport_b, protocol_b, port_b = await bind_udp_endpoint(host="127.0.0.1", prefer_prime=False, key=key, on_message=on_msg)
    transport_a, protocol_a, port_a = await bind_udp_endpoint(host="127.0.0.1", prefer_prime=False, key=key)
    try:
        send_envelope(transport_a, ("127.0.0.1", port_b), "node-a", "node-b", 1, 0, b"ping", key)
        for _ in range(50):
            if received:
                break
            await asyncio.sleep(0.02)
        assert received, "no UDP envelope received"
        assert received[0]["payload"] == b"ping"
        assert received[0]["sender"] == "node-a"
    finally:
        transport_a.close()
        transport_b.close()


In [ ]:
%%writefile meshnet_project/tests/test_beacon.py
import json
import urllib.request
from meshnet.mesh import MeshNetwork
from meshnet.web_beacon import start_beacon


def _get_json(url):
    with urllib.request.urlopen(url, timeout=3) as resp:
        return json.loads(resp.read().decode("utf-8"))


def _post_json(url, payload):
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=3) as resp:
        return json.loads(resp.read().decode("utf-8"))


def test_beacon_live_session_blockchain_and_map_ui():
    net = MeshNetwork()
    a = net.add_node("android-node", platform="android", udp_port=52417)
    b = net.add_node("terminal-node", platform="terminal", udp_port=53128)
    net.connect(a, b)
    net.hail(a)
    net.broadcast_common_chat(a, "beacon test message")

    beacon = start_beacon(net, host="127.0.0.1", port=0)
    try:
        assert beacon.url.startswith("http://") and ":0/" not in beacon.url
        with urllib.request.urlopen(beacon.url, timeout=3) as resp:
            body = resp.read().decode("utf-8")
        assert "Sensei Network Map" in body
        assert "Blockchain of messages" in body
        assert "COMPLETE PROJECT OBJECTIVE" in body or "Complete Objective" in body

        state = _get_json(beacon.url + "api/state")
        assert state["chain_valid"] is True
        assert len(state["chain"]) >= 2
        assert any(n["nickname"] == "android-node" for n in state["nodes"])

        # browser join pops in as new node and hails
        joined = _post_json(beacon.url + "api/join", {"platform": "web"})
        assert joined["ok"] is True
        assert "node" in joined and joined["node"]["nickname"]
        state2 = _get_json(beacon.url + "api/state")
        assert len(state2["nodes"]) >= len(state["nodes"]) + 1
        assert any("HAIL " + joined["node"]["nickname"] in b["payload"] for b in state2["chain"])
    finally:
        beacon.stop()


## Run tests


In [ ]:
# 3) Run full test suite
import os, sys, subprocess, pathlib

proj = pathlib.Path("meshnet_project").resolve()
env = os.environ.copy()
pp = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = str(proj) + (os.pathsep + pp if pp else "")

cmd = [sys.executable, "-m", "pytest", "tests", "-q"]
print("Running:", " ".join(cmd), "in", proj)
res = subprocess.run(cmd, cwd=str(proj), env=env, text=True)
if res.returncode != 0:
    raise SystemExit(res.returncode)
print("
All tests passed.")


## Live session: running simulation + testing link
Starts a live session beacon (OS-assigned port), seeds nodes, and runs a continuous background simulation that adds nodes/messages so the **Blockchain of messages** grows and new nodes **smoothly pop in** on the **Sensei Network Map**. Open the printed testing link in several browsers and click **Join as browser node**.

- In **Colab**, a proxied testing link is printed (when available) via the Colab kernel port proxy.
- In **local/Jupyter**, open the printed local URL.
- This cell starts background threads and does **not** block, so artifact generation and GitHub publish cells can run. The final cell keeps the session alive.


In [ ]:
# 4) Start live session simulation and print testing links
import sys, pathlib, time, random, threading

sys.path.insert(0, str(pathlib.Path("meshnet_project").resolve()))
from meshnet.mesh import MeshNetwork
from meshnet.port_selection import reserve_udp_port
from meshnet.web_beacon import start_beacon

random.seed(7)
LIVE_NET = MeshNetwork()

# Seed initial nodes with OS-assigned UDP ports (no fixed ports)
seed_specs = [("android-seed", "android"), ("terminal-seed", "terminal"), ("bash-seed", "bash")]
seed_nodes = []
for nick, plat in seed_specs:
    sock, port = reserve_udp_port(host="127.0.0.1", prefer_prime=False)
    sock.close()
    n = LIVE_NET.add_node(nick, platform=plat, udp_port=port, host="::1")
    seed_nodes.append(n)
LIVE_NET.connect(seed_nodes[0], seed_nodes[1])
LIVE_NET.connect(seed_nodes[1], seed_nodes[2])
for n in seed_nodes:
    LIVE_NET.hail(n)
LIVE_NET.create_group("common", [seed_nodes[0], seed_nodes[1], seed_nodes[2]])
LIVE_NET.broadcast_common_chat(seed_nodes[0], "Simulation online: two+ connected nodes suffice, no Internet required.")
LIVE_NET.broadcast_common_chat(seed_nodes[1], "Blockchain of messages active; open testing link in browsers to join and see Sensei Network Map pop-ins.")

# Start live beacon on OS-assigned port
LIVE_BEACON = start_beacon(LIVE_NET, host="0.0.0.0", port=0)
print("Local URL:", LIVE_BEACON.url)
print("Beacon port (OS-assigned):", LIVE_BEACON.port)

# Colab proxy/testing link
COLAB_TESTING_LINK = None
try:
    from google.colab.output import eval_js
    COLAB_TESTING_LINK = eval_js(f"google.colab.kernel.proxyPort({LIVE_BEACON.port})")
    print("
TESTING LINK (open in several browsers):", COLAB_TESTING_LINK)
except Exception as e:
    print("Colab proxy link unavailable:", e)
    print("
TESTING LINK (local):", LIVE_BEACON.url)

# Colab helpers
try:
    from google.colab import output
    try:
        output.serve_kernel_port_as_iframe(LIVE_BEACON.port)
    except Exception:
        pass
except Exception:
    pass

SIM_STOP = threading.Event()

def sim_loop():
    i = 0
    plats = ["android", "terminal", "bash", "web"]
    while not SIM_STOP.is_set():
        try:
            time.sleep(4.0)
            i += 1
            with LIVE_NET._lock:
                node_count = len(LIVE_NET.nodes)
            if node_count < 12 and (i % 2 == 1 or node_count < 5):
                plat = plats[i % len(plats)]
                nick = f"sim-{plat}-{i:02d}"
                try:
                    sock, port = reserve_udp_port(host="127.0.0.1", prefer_prime=False)
                    sock.close()
                except Exception:
                    port = None
                try:
                    n = LIVE_NET.add_node(nick, platform=plat, udp_port=port, host="::1")
                    others = [x for x in LIVE_NET.nodes.values() if x.node_id != n.node_id]
                    if others:
                        targets = random.sample(others, k=min(2, len(others)))
                        for t in targets:
                            LIVE_NET.connect(n, t)
                    LIVE_NET.hail(n)
                    LIVE_NET.broadcast_common_chat(n, f"{n.nickname} joined; peers={len(n.neighbors)}, udp_port={n.udp_port}")
                except Exception:
                    pass
            else:
                try:
                    nodes = list(LIVE_NET.nodes.values())
                    if len(nodes) >= 2:
                        s = random.choice(nodes)
                        LIVE_NET.broadcast_common_chat(s, f"heartbeat/proof from {s.nickname} (platform={s.platform}, udp_port={s.udp_port})")
                    sim_nodes = [n for n in nodes if n.nickname.startswith("sim-")]
                    if len(sim_nodes) >= 5 and i % 7 == 0:
                        victim = random.choice(sim_nodes)
                        try:
                            LIVE_NET.remove_node(victim)
                            speaker = random.choice(list(LIVE_NET.nodes.values())) if LIVE_NET.nodes else None
                            if speaker:
                                LIVE_NET.broadcast_common_chat(speaker, f"node {victim.nickname} disappeared; mesh rerouted/healed")
                        except Exception:
                            pass
                except Exception:
                    pass
        except Exception:
            time.sleep(2.0)

SIM_THREAD = threading.Thread(target=sim_loop, daemon=True)
SIM_THREAD.start()
print("
Live simulation running: nodes pop in on Sensei Network Map, hail, and append to Blockchain of messages.")
print("Open the testing link in several browsers and click 'Join as browser node'.")


## Generate production/download artifacts
Creates launchable/downloadable outputs in `dist/` and copies to `/artifacts/` when available:
- `meshnet.html` — standalone launchable HTML session client (same-origin live API or `?api=http://host:port`)
- `meshnet_terminal_installable.tar.gz` — terminal-installable production source package (`meshnet_project/`, installable via `pip install .`, `meshctl`)
- `meshnet_android_apk_project.zip` — buildable/launch-ready Android APK project scaffold (Gradle/Kotlin), bundled UI asset; build APK with Android SDK/JDK/Gradle
- `meshnet_release_artifacts.zip` — bundle of the above

> A compiled `.apk` binary requires Android toolchain; the complete buildable project and instructions are provided.


In [ ]:
# 5) Generate artifacts (HTML, terminal installable, Android APK project scaffold)
import sys, shutil, tarfile, zipfile, pathlib

ROOT = pathlib.Path.cwd()
PROJECT = ROOT / "meshnet_project"
DIST = ROOT / "dist"
ART = pathlib.Path("/artifacts")
DIST.mkdir(parents=True, exist_ok=True)
if ART.parent.exists():
    ART.mkdir(parents=True, exist_ok=True)

# 5a) Standalone launchable HTML (configurable API base via ?api=)
sys.path.insert(0, str(PROJECT.resolve()))
from meshnet.web_beacon import HTML_PAGE

standalone_html = HTML_PAGE.replace(
    "async function api(path, opts){",
    "async function api(path, opts){\n  const params=new URLSearchParams(location.search); const base=(params.get('api')||'').replace(/\\/$/,''); const url = path.startsWith('http')?path:(base+path);"
).replace(
    "  const r = await fetch(path, Object.assign(base, opts));",
    "  const r = await fetch(url, Object.assign(base, opts));"
).replace(
    '<div class="sub">No fixed ports · OS-assigned endpoint · 2 nodes suffice · No Internet required for mesh core · Live multi-browser join</div>',
    '<div class="sub">Launchable HTML client · connect to live session beacon (same-origin or ?api=http://host:port) · No fixed ports · 2 nodes suffice · Sensei Network Map + Blockchain of messages</div>'
)
(DIST / "meshnet.html").write_text(standalone_html, encoding="utf-8")
print("Wrote", DIST / "meshnet.html")

# 5b) Terminal installable production package
term_tar = DIST / "meshnet_terminal_installable.tar.gz"
with tarfile.open(term_tar, "w:gz") as tf:
    tf.add(PROJECT, arcname="meshnet_project")
print("Wrote", term_tar)

# 5c) Android APK build project scaffold (buildable/launch-ready)
android_dir = ROOT / "android_apk_project"
(app_main := android_dir / "app" / "src" / "main").mkdir(parents=True, exist_ok=True)
(android_dir / "app" / "src" / "main" / "java" / "com" / "meshnet").mkdir(parents=True, exist_ok=True)

(android_dir / "settings.gradle.kts").write_text('rootProject.name = "MeshNet"\ninclude(":app")\n', encoding="utf-8")
(android_dir / "build.gradle.kts").write_text(
    'plugins {\n'
    '    id("com.android.application") version "8.5.2" apply false\n'
    '    id("org.jetbrains.kotlin.android") version "1.9.24" apply false\n'
    '}\n', encoding="utf-8")
(android_dir / "gradle.properties").write_text(
    "org.gradle.jvmargs=-Xmx2g\nandroid.useAndroidX=true\nkotlin.code.style=official\n", encoding="utf-8")
(android_dir / "app" / "build.gradle.kts").write_text(
    'plugins {\n'
    '    id("com.android.application")\n'
    '    id("org.jetbrains.kotlin.android")\n'
    '}\n'
    'android {\n'
    '    namespace = "com.meshnet"\n'
    '    compileSdk = 34\n'
    '    defaultConfig {\n'
    '        applicationId = "com.meshnet"\n'
    '        minSdk = 26\n'
    '        targetSdk = 34\n'
    '        versionCode = 1\n'
    '        versionName = "0.2.0"\n'
    '    }\n'
    '    buildTypes {\n'
    '        release { isMinifyEnabled = false }\n'
    '        debug { isDebuggable = true }\n'
    '    }\n'
    '    compileOptions { sourceCompatibility = JavaVersion.VERSION_17; targetCompatibility = JavaVersion.VERSION_17 }\n'
    '    kotlinOptions { jvmTarget = "17" }\n'
    '}\n'
    'dependencies {\n'
    '    implementation("androidx.core:core-ktx:1.13.1")\n'
    '    implementation("androidx.appcompat:appcompat:1.7.0")\n'
    '    implementation("com.google.android.material:material:1.12.0")\n'
    '}\n', encoding="utf-8")

manifest = '''<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-permission android:name="android.permission.INTERNET" />
    <uses-permission android:name="android.permission.ACCESS_NETWORK_STATE" />
    <uses-permission android:name="android.permission.ACCESS_WIFI_STATE" />
    <uses-permission android:name="android.permission.CHANGE_WIFI_MULTICAST_STATE" />
    <uses-permission android:name="android.permission.BLUETOOTH" />
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" />
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT" />
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" />
    <uses-permission android:name="android.permission.BLUETOOTH_ADVERTISE" />
    <uses-permission android:name="android.permission.NEARBY_WIFI_DEVICES" />
    <uses-permission android:name="android.permission.ACCESS_FINE_LOCATION" />
    <uses-permission android:name="android.permission.FOREGROUND_SERVICE" />
    <application android:label="MeshNet" android:allowBackup="true" android:supportsRtl="true" android:theme="@style/Theme.AppCompat.Light.NoActionBar">
        <activity android:name=".MainActivity" android:exported="true">
            <intent-filter>
                <action android:name="android.intent.action.MAIN" />
                <category android:name="android.intent.category.LAUNCHER" />
            </intent-filter>
        </activity>
    </application>
</manifest>
'''
(android_dir / "app" / "src" / "main" / "AndroidManifest.xml").write_text(manifest, encoding="utf-8")

main_activity = r'''package com.meshnet

import android.os.Bundle
import android.webkit.WebSettings
import android.webkit.WebView
import androidx.appcompat.app.AppCompatActivity

/**
 * MeshNet Android client shell (launch-ready scaffold).
 * Protocol alignment: signed invites (QR/NFC/BLE/manual), one UDP socket per node on OS-assigned port,
 * compression-encrypted authenticated envelopes, replay protection, hail on connect, '?' complete objective/help,
 * routes/ping/chat/groups, Blockchain of messages, Sensei Network Map UI.
 * Extend with native UDP foreground service + QR/NFC/BLE bootstrap + local key storage.
 */
class MainActivity : AppCompatActivity() {
    override fun onCreate(savedInstanceState: Bundle?) {
        super.onCreate(savedInstanceState)
        val webView = WebView(this)
        val settings: WebSettings = webView.settings
        settings.javaScriptEnabled = true
        settings.domStorageEnabled = true
        setContentView(webView)
        // Bundled MeshNet UI (assets/meshnet.html). Set ?api=http://<session-host>:<port> for live session.
        webView.loadUrl("file:///android_asset/meshnet.html")
    }
}
'''
(android_dir / "app" / "src" / "main" / "java" / "com" / "meshnet" / "MainActivity.kt").write_text(main_activity, encoding="utf-8")

android_readme = r'''# MeshNet Android APK Build Project

Buildable/launch-ready scaffold aligned with the complete MeshNet objective:
- QR/NFC/BLE/manual-paste signed invite bootstrap
- One UDP socket per node on OS-assigned port; optional prime-port preference (49152-65535); no fixed port
- Two connected nodes suffice without Internet; no pi-sync/time-server chain
- Compression-encrypted authenticated envelopes, replay/sequence protection, local private keys
- Hail on connect: nickname, platform, node_id, udp_port/endpoint, peers
- `?` complete objective/help, chat/groups, routes/ping visualization
- Blockchain of messages (common chat) and Sensei Network Map responsive UI

## Build APK
Requires Android SDK, JDK 17+, and Gradle (wrapper recommended).

Commands:
    gradle wrapper        # if wrapper not present
    ./gradlew assembleDebug
    ./gradlew assembleRelease

Outputs (when built):
- app/build/outputs/apk/debug/app-debug.apk
- app/build/outputs/apk/release/app-release-unsigned.apk (sign for distribution)

The bundled UI asset (app/src/main/assets/meshnet.html) can connect to a live MeshNet session beacon via ?api=http://host:port.
'''
(android_dir / "README.md").write_text(android_readme, encoding="utf-8")

assets = android_dir / "app" / "src" / "main" / "assets"
assets.mkdir(parents=True, exist_ok=True)
shutil.copy2(DIST / "meshnet.html", assets / "meshnet.html")

apk_zip = DIST / "meshnet_android_apk_project.zip"
with zipfile.ZipFile(apk_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in android_dir.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(ROOT))
print("Wrote", apk_zip)

# 5d) bundle dist artifacts
dist_zip = DIST / "meshnet_release_artifacts.zip"
with zipfile.ZipFile(dist_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in ["meshnet.html", "meshnet_terminal_installable.tar.gz", "meshnet_android_apk_project.zip"]:
        zf.write(DIST / name, name)
print("Wrote", dist_zip)

# copy to /artifacts
if ART.parent.exists():
    ART.mkdir(parents=True, exist_ok=True)
    for p in [DIST / "meshnet.html", term_tar, apk_zip, dist_zip, ROOT / "meshnet_colab_envelope.ipynb"]:
        if p.exists():
            shutil.copy2(p, ART / p.name)
    print("Copied artifacts to", ART)

print("\nArtifacts ready:")
for p in sorted(DIST.iterdir()):
    print(" -", p.name, p.stat().st_size, "bytes")


## Publish to GitHub
Commits and pushes project materials on the current session branch. Existing `server/` work is preserved.


In [ ]:
# 6) Publish all to GitHub (commit and push on session branch)
import subprocess, os, pathlib

ROOT = pathlib.Path.cwd()
os.chdir(ROOT)

def run(cmd, check=False):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, text=True, capture_output=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if check and r.returncode != 0:
        raise SystemExit(r.returncode)
    return r

# Show context
run(["git", "rev-parse", "--abbrev-ref", "HEAD"])
run(["git", "remote", "-v"])

# Stage project outputs (preserve existing server/ untouched except not removing)
paths = [
    "meshnet_colab_envelope.ipynb",
    "meshnet_project",
    "dist",
    "android_apk_project",
]
existing = [p for p in paths if (ROOT / p).exists()]
if existing:
    run(["git", "add", "-A"] + existing, check=False)

run(["git", "status", "--short"])

# Commit if there are staged changes
diff = subprocess.run(["git", "diff", "--cached", "--quiet"], text=True)
if diff.returncode != 0:
    run(["git", "commit", "-m", "Add complete MeshNet Colab envelope, live simulation, blockchain chat, and release artifacts"], check=False)
else:
    print("No staged changes to commit.")

# Push current/session branch
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True, capture_output=True).stdout.strip()
if branch:
    print(f"\nPushing branch: {branch}")
    push = run(["git", "push", "-u", "origin", branch], check=False)
    if push.returncode != 0:
        print("\nPush did not complete successfully. If auth/permissions are unavailable in this environment, run the printed git commands locally or in an authenticated environment to publish.")
else:
    print("Could not determine current branch.")


## Keep session alive
Run this final cell to keep the live beacon/simulation running (serving the testing link). In **Runtime → Run all**, prior cells will have already generated artifacts and attempted GitHub publish.


In [ ]:
# 7) Keep session alive (run this cell to keep the live simulation/beacon running)
import time
print("Keeping live MeshNet session running. Press Interrupt/Stop to end.")
try:
    while True:
        time.sleep(5)
except KeyboardInterrupt:
    print("Session stopped.")


## Notes on production outputs
- **Testing link / multi-browser join:** open the printed URL in several browsers; each browser can "Join as browser node" to appear on the Sensei Network Map and participate in common chat. The Blockchain of messages view updates live via polling.
- **HTML:** `dist/meshnet.html` is launchable; for live session data use `?api=http://<host>:<port>` pointing to a running beacon (or open served by beacon).
- **Terminal installable:** extract `meshnet_terminal_installable.tar.gz`, then `pip install -e .` (or `pip install .`) inside `meshnet_project` and use `meshctl help|invite|serve-web|demo`.
- **Android APK:** extract `meshnet_android_apk_project.zip`, install Android SDK/JDK/Gradle, run `gradle wrapper` if needed, then `./gradlew assembleDebug` / `assembleRelease` to produce APK binaries.
- **Sandbox note:** this environment may not provide a publicly reachable browser URL; use the Colab-proxied testing link when running in Colab, or run locally for LAN/browser access.
